In [ ]:
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from tools import web_search_tool, save_report_to_md
from dotenv import load_dotenv

load_dotenv()

In [ ]:
model_client = OpenAIChatCompletionClient(
    model="gpt-4.1-nano-2025-04-14",
)

In [ ]:
research_planner = AssistantAgent(
    "research_planner",
    description="복잡한 질문을 연구 하위 과제로 분해하는 전략적 연구 코디네이터",
    model_client=model_client,
    system_message="""당신은 연구 기획 전문가입니다. 집중된 연구 계획을 수립하는 것이 당신의 역할입니다.

모든 응답은 반드시 한국어로 작성하세요.

각 연구 질문에 대해 다음을 포함한 집중된 연구 계획을 작성하세요:

1. **핵심 주제**: 조사할 주요 영역 2~3개
2. **검색 쿼리**: 다음을 포괄하는 구체적인 검색 쿼리 3~5개 작성
   - 최신 동향 및 뉴스
   - 핵심 통계 또는 데이터
   - 전문가 분석 또는 연구
   - 미래 전망

계획은 집중적이고 실행 가능하게 유지하세요. 양보다 질을 우선하세요.""",
)

research_agent = AssistantAgent(
    "research_agent",
    description="웹에서 검색하고 콘텐츠를 추출하는 웹 리서치 전문가",
    tools=[web_search_tool],
    model_client=model_client,
    system_message="""당신은 웹 리서치 전문가입니다. 연구 계획에 따라 집중적인 검색을 수행하는 것이 당신의 역할입니다.

모든 응답은 반드시 한국어로 작성하세요.

연구 전략:
1. **연구 계획에서 3~5회 검색** 실행
2. 검색 결과에서 **핵심 정보 추출**:
   - 주요 사실과 통계
   - 최근 동향
   - 전문가 의견
   - 중요한 맥락

3. **품질 기준**:
   - 신뢰할 수 있는 출처 우선
   - 최근 2년 이내 정보 탐색
   - 다양한 관점 기록

계획에 따른 검색을 완료한 후, 발견한 내용을 요약하세요. 목표는 5~10개의 고품질 출처를 수집하는 것입니다.""",
)

research_analyst = AssistantAgent(
    "research_analyst",
    description="연구 보고서를 작성하는 전문 분석가",
    model_client=model_client,
    system_message="""당신은 연구 분석가입니다. 수집된 연구 자료를 바탕으로 종합 보고서를 작성하세요.

모든 응답은 반드시 한국어로 작성하세요.

다음 구조의 연구 보고서를 작성하세요:

## 요약
- 핵심 발견 및 결론
- 주요 인사이트

## 배경 및 현황
- 현재 상황
- 최근 동향
- 핵심 통계 및 데이터

## 분석 및 인사이트
- 주요 트렌드
- 다양한 관점
- 전문가 의견

## 미래 전망
- 떠오르는 트렌드
- 예측
- 시사점

## 출처
- 사용한 모든 출처 목록

수집된 연구를 바탕으로 명확하고 잘 구조화된 보고서를 작성하세요. 완료 시 반드시 "REPORT_COMPLETE"로 끝내세요.""",
)

quality_reviewer = AssistantAgent(
    "quality_reviewer",
    description="연구의 완성도와 정확성을 평가하는 품질 보증 전문가",
    tools=[save_report_to_md],
    model_client=model_client,
    system_message="""당신은 품질 검토자입니다. 연구 분석가가 완전한 연구 보고서를 작성했는지 확인하는 것이 당신의 역할입니다.

모든 응답은 반드시 한국어로 작성하세요.

다음을 확인하세요:
- 연구 분석가가 작성한 종합 보고서가 "REPORT_COMPLETE"로 끝나는지
- 연구 질문이 충분히 답변되었는지
- 출처가 인용되었고 신뢰할 수 있는지
- 보고서에 요약, 핵심 정보, 분석, 출처가 포함되어 있는지

"REPORT_COMPLETE"로 끝나는 완전한 연구 보고서를 확인하면:
1. 먼저 save_report_to_md 도구를 사용해 보고서를 report.md에 저장하세요
2. 그다음 다음과 같이 말하세요: "연구가 완료되었습니다. 보고서가 report.md에 저장되었습니다. 보고서를 검토하시고 승인하시거나 추가 연구가 필요한지 알려주세요."

연구 분석가가 아직 완전한 보고서를 작성하지 않았다면, 지금 작성하도록 요청하세요.""",
)

research_enhancer = AssistantAgent(
    "research_enhancer",
    description="치명적인 공백만 식별하는 전문가",
    model_client=model_client,
    system_message="""당신은 연구 보완 전문가입니다. 오직 치명적인 공백만 식별하는 것이 당신의 역할입니다.

모든 응답은 반드시 한국어로 작성하세요.

연구를 검토하고, 다음과 같은 중대한 공백이 있을 때만 추가 검색을 제안하세요:
- 최근 6개월 동향이 완전히 누락됨
- 통계나 데이터가 전혀 없음
- 요청된 핵심 관점이 빠져 있음

연구가 기본 사항을 합리적으로 다루고 있다면, 다음과 같이 말하세요: "연구가 충분하여 보고서 작성을 진행할 수 있습니다."

절대적으로 필요한 경우에만 추가 검색 1~2개를 제안하세요. 완벽한 커버리지보다 좋은 보고서를 완성하는 것을 우선합니다.""",
)

user_proxy = UserProxyAgent(
    "user_proxy",
    description="추가 연구를 요청하거나 최종 결과를 승인할 수 있는 인간 검토자",
    input_func=input,
)

In [ ]:
selector_prompt = """
대화 기록을 바탕으로 현재 작업에 가장 적합한 에이전트를 선택하세요:

{roles}

현재 대화:
{history}

사용 가능한 에이전트:
- research_planner: 연구 접근 방식 기획 (시작 시에만)
- research_agent: 웹 소스 검색 및 콘텐츠 추출 (기획 완료 후)
- research_enhancer: 치명적인 공백만 식별 (최소한으로 사용)
- research_analyst: 최종 연구 보고서 작성
- quality_reviewer: 완전한 보고서 존재 여부 확인
- user_proxy: 사용자에게 피드백 요청

워크플로:
1. 아직 기획이 안 됐다면 → research_planner 선택
2. 기획은 됐지만 연구가 안 됐다면 → research_agent 선택
3. research_agent가 초기 검색을 완료한 후 → research_enhancer를 한 번 선택
4. enhancer가 "연구가 충분하여 보고서 작성을 진행할 수 있습니다"라고 말하면 → research_analyst 선택
5. enhancer가 중요한 추가 검색을 제안하면 → research_agent를 한 번 더 선택한 후 research_analyst 선택
6. research_analyst가 "REPORT_COMPLETE"라고 말했으면 → quality_reviewer 선택
7. quality_reviewer가 사용자 피드백을 요청했으면 → user_proxy 선택

중요: research_agent가 최대 2번 검색한 후에는, 상황과 관계없이 research_analyst로 진행하세요.

이 워크플로에 따라 다음에 작업할 에이전트를 선택하세요."""

In [ ]:
text_termination = TextMentionTermination("APPROVED")
max_message_termination = MaxMessageTermination(max_messages=50)
termination_condition = text_termination | max_message_termination

team = SelectorGroupChat(
    participants=[
        research_agent,
        research_analyst,
        research_enhancer,
        research_planner,
        quality_reviewer,
        user_proxy,
    ],
    selector_prompt=selector_prompt,
    model_client=model_client,
    # allow_repeated_speaker=True,
    termination_condition=termination_condition,
)

In [ ]:
await Console(
    team.run_stream(task="AI에 친숙하지 않은 사람에게 AI의 변화 흐름을 설명할 수 있는 방법에 대해서 조사해봐")
)